In [ ]:
%gherkin
Feature: Masking invoice numbers in d_product_revenue_clone

  Background:
    Given a connection to Unity Catalog with catalog purgo_databricks in schema purgo_playground
    And d_product_revenue table exists with invoice_number of type bigint
    And necessary permissions to drop and create tables

  Scenario: Apply masking to invoice_number in d_product_revenue_clone
    Given the d_product_revenue_clone table does not exist
    When I create a replica of the d_product_revenue table named d_product_revenue_clone
    Then the d_product_revenue_clone table should be successfully created
    When I cast invoice_number to string for processing
    And I mask the last 4 digits of invoice_number by replacing them with '****'
    Then the masked invoice_number should be updated in the d_product_revenue_clone table
    And the masking should be successfully completed

  Scenario Outline: Handling null or malformed invoice numbers
    Given the d_product_revenue_clone table with null or malformed <invoice_number>
    When I attempt to mask the last 4 digits of <invoice_number>
    Then an error message "<error_message>" should be logged

    Examples:
      | invoice_number | error_message                      |
      | NULL           | "Invoice number is null"           |
      | 'abcd'         | "Invoice number is malformed"      |
  
  Scenario: Error when required permissions are missing
    Given a user with insufficient permissions
    When I attempt to drop the d_product_revenue_clone table
    Then an error message "Insufficient permissions to drop table" should be returned
